# 02 · The candidate pool

The 21 predictors of notebook 01 were chosen by reasoning about antenatal care. This notebook
asks the survey instead: every one of its 6,648 columns is screened against ANC8+ under the
survey design, and the admissible ones are added to the 21 to form the **candidate pool**.

Screening ranks, it does not select. A column is admitted on three grounds, in this order:

1. **Timing.** It must be known before antenatal care begins. Anything recorded at or after
   the delivery, and anything the antenatal care itself changes, is refused, however well it
   predicts. The refused variables and the reason for each are listed below; `v394` (visited a
   health facility in the last 12 months) is the clearest case, because an antenatal contact
   *is* a facility visit.
2. **Coverage.** Observed for at least half the women. Most of the file is birth-history
   slots filled for a handful of women.
3. **Meaning.** Whole blocks (wealth, education, partner, household, media, access,
   empowerment, barriers to care) are admitted together, with a few derived measures such as
   spousal gaps, counts and the log of travel time.

**Reads:** `data/ETIR8AFL.DTA`, `outputs/ANC8_analysis_frozen.pkl`
**Writes:** `outputs/enhanced/ANC8_analysis_frozen_v2.pkl` (the pool frame),
`feature_spec_v2.json`, `candidate_screening.csv`, `predictor_register_v2.csv`, `birth_timing.parquet`

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

from anc8 import cohorts, features, screening
OUT = P.outputs / "enhanced"
OUT.mkdir(parents=True, exist_ok=True)

## The cohort, rebuilt from the survey file

Women on the maternal-health roster with a resolved contact count (`m14_1` from 0 to 97;
98, "don't know", is unresolved). The rebuilt cohort must be the same women as notebook 01's.

In [ ]:
reader = pd.read_stata(cohorts.IR_FILE, iterator=True)
VARIABLE_LABELS = reader.variable_labels()
ir = pd.read_stata(cohorts.IR_FILE, convert_categoricals=False)
print("survey file:", ir.shape[0], "women x", ir.shape[1], "columns")

cohort = ir[ir.m14_1.notna() & ir.m14_1.between(0, 97)].copy()
y = (cohort.m14_1 >= 8).astype(int).to_numpy()
w = (cohort.v005 / 1e6).to_numpy()
first = pd.read_pickle(P.outputs / "ANC8_analysis_frozen.pkl")
first = first[first.anc8plus.notna()].copy()
repro.check("the cohort is notebook 01's, woman for woman", set(cohort.caseid) == set(first.caseid), f"{len(cohort):,} women")
print(f"weighted ANC8+ prevalence: {np.average(y, weights=w):.1%}")

## Screening every column

For each column: the survey-weighted AUC of the raw variable as a score (which assumes the
variable is ordered), the spread of ANC8+ prevalence across its categories (which catches
unordered variables such as region), and how often it is missing.

In [ ]:
screen = screening.screen(cohort, VARIABLE_LABELS, y, w)
screen.to_csv(OUT / "candidate_screening.csv", index=False)
print(screen.screen_status.value_counts().to_string())
print("\nThe 20 strongest columns observed for at least half the women:")
print(screen[screen.screen_status == "candidate"].head(20)[["variable", "label", "missing", "weighted_auc", "prevalence_spread"]].to_string(index=False))

## Refused on timing

Variables that predict well because they partly record the antenatal care itself.

In [ ]:
for var, reason in features.TEMPORAL_EXCLUSIONS.items():
    auc = screen.loc[screen.variable == var, "weighted_auc"]
    print(f"{var:8s} AUC {auc.iloc[0] if len(auc) and pd.notna(auc.iloc[0]) else float('nan'):.3f}  {VARIABLE_LABELS.get(var, '')}")
    print(f"         {reason}")

## Assembling the pool

The 21 predictors of notebook 01 are carried over unchanged (their careful handling of
"not applicable" stays), the admitted survey variables are added, and a few measures are
derived: spousal education and age gaps, counts of decisions, of attitudes and of barriers,
an asset index, and travel time treated as a log with flags for zero, top-coding and missing.
Travel time to a health post is missing for about a quarter of women by design (no post in
range), so it carries a `healthpost_time_missing` flag instead of an invented value.

In [ ]:
pool_frame, numeric, categorical = features.assemble(first, ir)
print(f"pool frame: {pool_frame.shape[0]:,} women x {pool_frame.shape[1]} columns")
print(f"{len(numeric)} numeric and {len(categorical)} categorical predictors: {len(numeric) + len(categorical)} in all")

# the leakage audit: nothing from the delivery or after, nothing refused, no outcome or design variable
FORBIDDEN = ("m13", "m14", "m15", "m16", "m17", "m18", "m19")
predictors = set(numeric) | set(categorical)
repro.check("no delivery-era variable among the predictors", not [c for c in predictors if c.lower().startswith(FORBIDDEN)])
repro.check("no variable refused on timing", not (set(features.TEMPORAL_EXCLUSIONS) & predictors))
repro.check("no outcome or survey-design variable", not ({"anc1plus", "anc4plus", "anc8plus", "v005", "v021", "v022"} & predictors))

In [ ]:
pool_frame.to_pickle(OUT / "ANC8_analysis_frozen_v2.pkl")
arch = json.load(open(P.outputs / "notebook3_explainability_handoff" / "predictor_architecture.json"))
spec = {"numeric_predictors": numeric, "categorical_predictors": categorical,
        "n_predictors": len(numeric) + len(categorical), "n_original_predictors": len(arch["full_cohort_predictors"]),
        "outcome": "anc8plus", "survey_design": {"weight": "v005", "PSU": "v021", "stratum": "v022"},
        "temporal_exclusions": features.TEMPORAL_EXCLUSIONS}
json.dump(spec, open(OUT / "feature_spec_v2.json", "w"), indent=2)

# the timing of each woman's most recent birth (for age at the birth and the 24-month cohort)
timing = cohorts.birth_timing()
print("birth timing for", len(timing), "women")

## The women who started antenatal care, and when

One lever of notebook 13, an earlier first visit, needs to know when care began. The survey
records the timing of the first contact in weeks (`m13a_1` from 100) or months (from 200). A
first contact at 12 weeks or earlier, or in the first three months, is **early**; later is
not; "don't know" (998) and women with no contact stay unresolved and are left out. These
5,001 women are the ANC-entry cohort, on the same split as everyone else.

In [ ]:
t = ir[["caseid", "m13a_1"]].copy()
t["caseid"] = t.caseid.astype(str)
weeks, months = t.m13a_1.between(100, 199), t.m13a_1.between(200, 299)
t["early_anc"] = np.nan
t.loc[weeks, "early_anc"] = (t.m13a_1[weeks] - 100 <= 12).astype(float)
t.loc[months, "early_anc"] = (t.m13a_1[months] - 200 <= 3).astype(float)
full = cohorts.load("full")[["caseid", "partition"]]
entry = full.merge(t[t.early_anc.notna()], on="caseid", how="inner")
entry = entry.rename(columns={"partition": "entry_partition"})[["caseid", "early_anc", "entry_partition"]]
entry.to_csv(OUT / "anc_entry.csv", index=False)
print(f"{len(entry):,} women started care with a known timing; {entry.early_anc.mean():.0%} of them early (unweighted)")

## Checks against the study

In [ ]:
ref = repro.reference("fingerprints.json")
repro.check("95 predictors", spec["n_predictors"] == 95)
e = cohorts.entry_table().sort_values("caseid").reset_index(drop=True)
repro.check("the ANC-entry cohort is the study's (content fingerprint)", repro.fingerprint(e) == ref["anc_entry"], f"{len(e):,} women")
repro.check("the pool frame is identical to the study's (content fingerprint)",
            repro.fingerprint(pool_frame) == ref["analysis_frame_pool"], repro.fingerprint(pool_frame))
repro.report()